# AISC DeepFake - 3-Bölge VGG16+SVM Late Fusion

**Amaç:** Daha önce eğitilmiş üç bölgesel modelin mevcut **test tahminlerini** birleştirerek tek bir nihai `REAL / FAKE` sonucu üretmek.

Bu notebook **yeni model eğitmez**, `.fit()` çağırmaz, feature extractor çalıştırmaz ve kaynak modellerin dosyalarını değiştirmez.

## Kullanılan tam kaynak sonuçlar

Notebook Drive klasörlerini **isimleriyle** takip eder; hiçbir Drive ID'si sabitlenmemiştir.

- Nazlıcan / Kaş: `VGG16_FeatureExtractor_SVM_Kas`
- Dilara / Ağız: `20260808_1240_mouth_vgg16_svm_seed42`
- Kader / Göz: `20260808_1124_eye_vgg16_svm_seed42`

Drive hiyerarşisi:

`MyDrive / AISC DeepFake Çalışmaları / Deney 1 / <Kişi> / Deney 1 / Sonuçlar / <run>`

Nihai füzyon çıktısı:

`MyDrive / AISC DeepFake Çalışmaları / Deney 1 / Nazlıcan / Deney 1 / füzyon sonuçları / <run_id>`

## Füzyon yöntemi

Kaynak sonuçlar incelendiğinde Kaş ve Göz sonuçlarında `fake_probability` bulunurken, Ağız sonucunda yalnızca mevcut SVM `decision_score_fake` ve sınıf kararı bulunuyor. Üç modeli yeniden eğitmeden veya yeni bir kalibrasyon modeli kurmadan üç adet karşılaştırılabilir olasılık elde etmek mümkün değil.

Bu nedenle notebook **mevcut üç model kararını eşit ağırlıkla birleştiren 3-yollu Majority-Vote Late Fusion** kullanır:

`fusion_vote_score = (pred_kaş + pred_ağız + pred_göz) / 3`

- `0 = REAL`
- `1 = FAKE`
- En az 2 model `FAKE` derse nihai sonuç `FAKE`.
- Yeni model, meta-model veya kalibrasyon eğitimi yoktur.
- Kaynak olasılık/decision skorları denetim için saklanır; ölçekleri eşit olmadığı için birbirleriyle ham olarak ortalanmaz.

In [1]:
# 1) Google Drive'i bağla ve temel kütüphaneleri yükle
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import unicodedata
import hashlib
import json
import os
import platform
import re
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn import __version__ as sklearn_version
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
POSITIVE_CLASS = 1
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn_version)

Mounted at /content/drive
Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.2.2
scikit-learn: 1.6.1


In [2]:
# 2) Isim bazli ve Unicode-guvenli Drive yol çözümleyicileri
# Drive ID kullanilmaz. Türkçe karakterlerin composed/decomposed farklari normalize edilir.

MYDRIVE = Path("/content/drive/MyDrive")

def norm_name(value: str) -> str:
    return unicodedata.normalize("NFKC", str(value)).strip().casefold()

def resolve_child(parent: Path, wanted_name: str, expect_dir: bool = True) -> Path:
    if not parent.exists():
        raise FileNotFoundError(f"Parent klasör bulunamadi: {parent}")

    matches = []
    for p in parent.iterdir():
        if norm_name(p.name) == norm_name(wanted_name):
            if expect_dir and p.is_dir():
                matches.append(p)
            elif (not expect_dir) and p.is_file():
                matches.append(p)

    if len(matches) == 0:
        available = sorted(x.name for x in parent.iterdir())
        raise FileNotFoundError(
            f"'{wanted_name}' bulunamadi.\nParent: {parent}\nMevcut ögeler: {available}"
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Belirsiz yol: '{wanted_name}' için birden fazla eslesme var: {matches}"
        )
    return matches[0]

def find_files_recursive(root: Path, wanted_name: str) -> list[Path]:
    return sorted(
        [
            p for p in root.rglob("*")
            if p.is_file() and norm_name(p.name) == norm_name(wanted_name)
        ],
        key=lambda p: str(p),
    )

def select_csv_by_schema(root: Path, filename: str, required_columns: set[str]) -> Path:
    candidates = find_files_recursive(root, filename)
    valid = []
    rejected = {}

    for p in candidates:
        try:
            cols = set(pd.read_csv(p, nrows=2, encoding="utf-8-sig").columns)
            if required_columns.issubset(cols):
                valid.append(p)
            else:
                rejected[str(p)] = sorted(required_columns - cols)
        except Exception as exc:
            rejected[str(p)] = [f"READ_ERROR: {exc}"]

    if len(valid) != 1:
        raise RuntimeError(
            f"{root} içinde tek ve dogru '{filename}' bulunamadi.\n"
            f"Valid: {valid}\nRejected: {json.dumps(rejected, ensure_ascii=False, indent=2)}"
        )
    return valid[0]

def select_npz_by_keys(root: Path, filename: str, required_keys: set[str]) -> Path:
    candidates = find_files_recursive(root, filename)
    valid = []
    rejected = {}

    for p in candidates:
        try:
            with np.load(p, allow_pickle=True) as z:
                keys = set(z.files)
            if required_keys.issubset(keys):
                valid.append(p)
            else:
                rejected[str(p)] = sorted(required_keys - keys)
        except Exception as exc:
            rejected[str(p)] = [f"READ_ERROR: {exc}"]

    if len(valid) != 1:
        raise RuntimeError(
            f"{root} içinde tek ve dogru '{filename}' bulunamadi.\n"
            f"Valid: {valid}\nRejected: {json.dumps(rejected, ensure_ascii=False, indent=2)}"
        )
    return valid[0]

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("Yol çözümleyiciler hazir.")

Yol çözümleyiciler hazir.


In [3]:
# 3) Tam olarak kullanilacak Drive klasörlerini ISIMLERIYLE takip et
# Gerçek hiyerarsi:
# AISC DeepFake Çalışmaları / Deney 1 / Kisi / Deney 1

PROJECT = resolve_child(MYDRIVE, "AISC DeepFake Çalışmaları")
OUTER_EXPERIMENT = resolve_child(PROJECT, "Deney 1")

NAZLICAN = resolve_child(OUTER_EXPERIMENT, "Nazlıcan")
DILARA = resolve_child(OUTER_EXPERIMENT, "Dilara")
KADER = resolve_child(OUTER_EXPERIMENT, "Kader")

NAZLICAN_EXP1 = resolve_child(NAZLICAN, "Deney 1")
DILARA_EXP1 = resolve_child(DILARA, "Deney 1")
KADER_EXP1 = resolve_child(KADER, "Deney 1")

NAZLICAN_RESULTS = resolve_child(NAZLICAN_EXP1, "Sonuçlar")
DILARA_RESULTS = resolve_child(DILARA_EXP1, "Sonuçlar")
KADER_RESULTS = resolve_child(KADER_EXP1, "Sonuçlar")

EYEBROW_RUN_NAME = "VGG16_FeatureExtractor_SVM_Kas"
MOUTH_RUN_NAME = "20260808_1240_mouth_vgg16_svm_seed42"
EYE_RUN_NAME = "20260808_1124_eye_vgg16_svm_seed42"

EYEBROW_RUN = resolve_child(NAZLICAN_RESULTS, EYEBROW_RUN_NAME)
MOUTH_RUN = resolve_child(DILARA_RESULTS, MOUTH_RUN_NAME)
EYE_RUN = resolve_child(KADER_RESULTS, EYE_RUN_NAME)

FUSION_ROOT = resolve_child(NAZLICAN_EXP1, "füzyon sonuçları")

print("SOURCE 1 - EYEBROW:", EYEBROW_RUN)
print("SOURCE 2 - MOUTH  :", MOUTH_RUN)
print("SOURCE 3 - EYE    :", EYE_RUN)
print("TARGET ROOT       :", FUSION_ROOT)

assert EYEBROW_RUN != FUSION_ROOT
assert MOUTH_RUN != FUSION_ROOT
assert EYE_RUN != FUSION_ROOT

SOURCE 1 - EYEBROW: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/VGG16_FeatureExtractor_SVM_Kas
SOURCE 2 - MOUTH  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260808_1240_mouth_vgg16_svm_seed42
SOURCE 3 - EYE    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260808_1124_eye_vgg16_svm_seed42
TARGET ROOT       : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları


In [4]:
# 4) Her run içindeki dogru prediction / metadata dosyalarini semaya göre bul
# Sabit alt-klasör varsayimi yapilmaz; dosya adi + zorunlu sütun/anahtar semasi dogrulanir.

eyebrow_pred_path = select_csv_by_schema(
    EYEBROW_RUN,
    "test_predictions.csv",
    {
        "video_id", "true_label", "predicted_label",
        "fake_probability", "decision_score"
    },
)

mouth_pred_path = select_csv_by_schema(
    MOUTH_RUN,
    "test_predictions.csv",
    {
        "true_label_id", "predicted_label_id", "decision_score_fake"
    },
)

mouth_feature_path = select_npz_by_keys(
    MOUTH_RUN,
    "test_features.npz",
    {"labels", "sample_ids", "video_ids", "paths", "class_names"},
)

eye_pred_path = select_csv_by_schema(
    EYE_RUN,
    "test_predictions.csv",
    {
        "sample_id", "image_path", "true_label", "predicted_label",
        "fake_probability", "decision_score", "run_id"
    },
)

print("Eyebrow predictions:", eyebrow_pred_path)
print("Mouth predictions   :", mouth_pred_path)
print("Mouth test metadata :", mouth_feature_path)
print("Eye predictions     :", eye_pred_path)

source_files = {
    "eyebrow_predictions": eyebrow_pred_path,
    "mouth_predictions": mouth_pred_path,
    "mouth_test_features": mouth_feature_path,
    "eye_predictions": eye_pred_path,
}

Eyebrow predictions: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/VGG16_FeatureExtractor_SVM_Kas/predictions/test_predictions.csv
Mouth predictions   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260808_1240_mouth_vgg16_svm_seed42/artifacts/test_predictions.csv
Mouth test metadata : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260808_1240_mouth_vgg16_svm_seed42/artifacts/test_features.npz
Eye predictions     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260808_1124_eye_vgg16_svm_seed42/predictions/test_predictions.csv


In [5]:
# 5) Kaynak tahminleri oku ve HERHANGI BIR MODEL EGITMEDEN ortak sample kimligini olustur

eyebrow = pd.read_csv(eyebrow_pred_path, encoding="utf-8-sig")
mouth = pd.read_csv(mouth_pred_path, encoding="utf-8-sig")
eye = pd.read_csv(eye_pred_path, encoding="utf-8-sig")

with np.load(mouth_feature_path, allow_pickle=True) as z:
    mouth_npz = {k: z[k].copy() for k in z.files}

# Mouth CSV -> NPZ row-order provenance gate
n_mouth = len(mouth)
assert len(mouth_npz["labels"]) == n_mouth, (
    f"Mouth CSV/NPZ satir sayisi farkli: CSV={n_mouth}, NPZ={len(mouth_npz['labels'])}"
)

if "sample_id" in mouth.columns:
    expected_generic = [f"test_sample_{i:05d}" for i in range(n_mouth)]
    assert mouth["sample_id"].astype(str).tolist() == expected_generic, (
        "Mouth prediction satir sirasi beklenen test_sample_XXXXX dizisiyle uyusmuyor. "
        "Kimlikleri tahmin ederek birlestirmek yerine süreç durduruldu."
    )

npz_labels = np.asarray(mouth_npz["labels"]).astype(int)
csv_labels = mouth["true_label_id"].to_numpy(dtype=int)
assert np.array_equal(npz_labels, csv_labels), (
    "Mouth test_features.npz ile test_predictions.csv true-label sirasi uyusmuyor."
)

mouth["source_sample_id"] = np.asarray(mouth_npz["sample_ids"]).astype(str)
mouth["source_video_id"] = np.asarray(mouth_npz["video_ids"]).astype(str)
mouth["source_roi_path"] = np.asarray(mouth_npz["paths"]).astype(str)
mouth["frame_key"] = mouth["source_video_id"]

mouth_face = mouth["source_sample_id"].str.extract(r"face(\d+)$", expand=False)
assert mouth_face.notna().all(), "Mouth face_index parse edilemeyen sample var."
mouth["face_index"] = mouth_face.astype(int)

# Eye canonical frame / face
eye["frame_key"] = eye["image_path"].astype(str).str.extract(
    r"((?:real|fake)_test_\d+)", expand=False
)
eye_face = eye["image_path"].astype(str).str.extract(r"__face_(\d+)", expand=False)
assert eye["frame_key"].notna().all(), "Eye frame_key parse edilemeyen satir var."
assert eye_face.notna().all(), "Eye face_index parse edilemeyen satir var."
eye["face_index"] = eye_face.astype(int)

# Eyebrow canonical frame
eyebrow["frame_key"] = eyebrow["video_id"].astype(str)
assert eyebrow["frame_key"].str.match(r"^(?:real|fake)_test_\d+$").all(), (
    "Eyebrow video_id beklenen test frame kimligi formatinda degil."
)
eyebrow["face_index"] = 0

for df, label_col, name in [
    (eyebrow, "true_label", "eyebrow"),
    (mouth, "true_label_id", "mouth"),
    (eye, "true_label", "eye"),
]:
    vals = set(pd.to_numeric(df[label_col], errors="raise").astype(int).unique())
    assert vals.issubset({0, 1}), f"{name} etiketleri 0/1 degil: {vals}"

print("Raw counts:")
print("  Eyebrow:", len(eyebrow))
print("  Mouth  :", len(mouth))
print("  Eye    :", len(eye))

Raw counts:
  Eyebrow: 196
  Mouth  : 302
  Eye    : 302


In [6]:
# 6) AYNI YÜZÜ temsil eden satirlari eslestir
# Kas kaydinda face_index yok ve tek ROI/frame var.
# Bu nedenle Eye ve Mouth tarafinda face_index == 0 kullanilir.
# Diger yüzler silinmez; audit'e kaydedilir.

mouth_extra_faces = mouth.loc[mouth["face_index"] != 0].copy()
eye_extra_faces = eye.loc[eye["face_index"] != 0].copy()

mouth0 = mouth.loc[mouth["face_index"] == 0].copy()
eye0 = eye.loc[eye["face_index"] == 0].copy()

assert eyebrow["frame_key"].is_unique, "Eyebrow frame_key duplicate var."
assert mouth0["frame_key"].is_unique, "Mouth face0 frame_key duplicate var."
assert eye0["frame_key"].is_unique, "Eye face0 frame_key duplicate var."

eyebrow_keys = set(eyebrow["frame_key"])
mouth_keys = set(mouth0["frame_key"])
eye_keys = set(eye0["frame_key"])

missing_in_mouth = sorted(eyebrow_keys - mouth_keys)
missing_in_eye = sorted(eyebrow_keys - eye_keys)

assert not missing_in_mouth, (
    f"Eyebrow test sample'larinin {len(missing_in_mouth)} tanesi Mouth'ta yok. "
    f"Ilk örnekler: {missing_in_mouth[:10]}"
)
assert not missing_in_eye, (
    f"Eyebrow test sample'larinin {len(missing_in_eye)} tanesi Eye'da yok. "
    f"Ilk örnekler: {missing_in_eye[:10]}"
)

eyebrow_s = eyebrow[
    [
        "frame_key", "true_label", "predicted_label",
        "fake_probability", "decision_score", "image_path"
    ]
].rename(columns={
    "true_label": "true_label_eyebrow",
    "predicted_label": "pred_eyebrow",
    "fake_probability": "eyebrow_fake_probability",
    "decision_score": "eyebrow_decision_score",
    "image_path": "eyebrow_image_path",
})

mouth_s = mouth0[
    [
        "frame_key", "true_label_id", "predicted_label_id",
        "decision_score_fake", "source_sample_id", "source_roi_path"
    ]
].rename(columns={
    "true_label_id": "true_label_mouth",
    "predicted_label_id": "pred_mouth",
    "decision_score_fake": "mouth_decision_score_fake",
    "source_sample_id": "mouth_source_sample_id",
    "source_roi_path": "mouth_image_path",
})

eye_s = eye0[
    [
        "frame_key", "true_label", "predicted_label",
        "fake_probability", "decision_score", "sample_id", "image_path"
    ]
].rename(columns={
    "true_label": "true_label_eye",
    "predicted_label": "pred_eye",
    "fake_probability": "eye_fake_probability",
    "decision_score": "eye_decision_score",
    "sample_id": "eye_source_sample_id",
    "image_path": "eye_image_path",
})

fused = (
    eyebrow_s
    .merge(mouth_s, on="frame_key", how="inner", validate="one_to_one")
    .merge(eye_s, on="frame_key", how="inner", validate="one_to_one")
)

assert len(fused) == len(eyebrow), (
    f"Eslesme kaybi var: eyebrow={len(eyebrow)}, fused={len(fused)}"
)

assert (fused["true_label_eyebrow"] == fused["true_label_mouth"]).all(), (
    "Eyebrow ve Mouth ground-truth etiketleri uyusmuyor."
)
assert (fused["true_label_eyebrow"] == fused["true_label_eye"]).all(), (
    "Eyebrow ve Eye ground-truth etiketleri uyusmuyor."
)

fused["true_label"] = fused["true_label_eyebrow"].astype(int)

for c in ["pred_eyebrow", "pred_mouth", "pred_eye"]:
    fused[c] = pd.to_numeric(fused[c], errors="raise").astype(int)
    assert set(fused[c].unique()).issubset({0, 1}), f"{c} 0/1 degil."

print("MATCHED COMMON TEST COHORT:", len(fused))
print("Excluded non-zero faces - Mouth:", len(mouth_extra_faces))
print("Excluded non-zero faces - Eye  :", len(eye_extra_faces))

MATCHED COMMON TEST COHORT: 196
Excluded non-zero faces - Mouth: 10
Excluded non-zero faces - Eye  : 10


In [7]:
# 7) ESIT AGIRLIKLI 3-YOLLU LATE FUSION - YENI MODEL YOK

fused["fake_vote_count"] = (
    fused["pred_eyebrow"] +
    fused["pred_mouth"] +
    fused["pred_eye"]
)

fused["fusion_vote_score"] = fused["fake_vote_count"] / 3.0
fused["fusion_predicted_label"] = (fused["fake_vote_count"] >= 2).astype(int)
fused["fusion_predicted_class"] = np.where(
    fused["fusion_predicted_label"] == 1, "FAKE", "REAL"
)
fused["true_class"] = np.where(fused["true_label"] == 1, "FAKE", "REAL")
fused["correct"] = fused["fusion_predicted_label"] == fused["true_label"]

y_true = fused["true_label"].to_numpy(dtype=int)
y_pred = fused["fusion_predicted_label"].to_numpy(dtype=int)
vote_score = fused["fusion_vote_score"].to_numpy(dtype=float)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fusion_metrics = {
    "n_test_samples": int(len(fused)),
    "fusion_method": "equal_weight_3way_majority_vote_late_fusion",
    "no_model_training": True,
    "positive_class": "FAKE",
    "weights": {"eyebrow": 1/3, "mouth": 1/3, "eye": 1/3},
    "decision_rule": "FAKE if at least 2 of 3 regional models predict FAKE",
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "precision_fake": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall_fake": float(recall_score(y_true, y_pred, zero_division=0)),
    "specificity_real": float(tn / (tn + fp)) if (tn + fp) else None,
    "f1_fake": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc_vote_score": float(roc_auc_score(y_true, vote_score)),
    "average_precision_vote_score": float(
        average_precision_score(y_true, vote_score)
    ),
    "mcc": float(matthews_corrcoef(y_true, y_pred)),
    "confusion_matrix": {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    },
}

print(json.dumps(fusion_metrics, indent=2, ensure_ascii=False))

{
  "n_test_samples": 196,
  "fusion_method": "equal_weight_3way_majority_vote_late_fusion",
  "no_model_training": true,
  "positive_class": "FAKE",
  "weights": {
    "eyebrow": 0.3333333333333333,
    "mouth": 0.3333333333333333,
    "eye": 0.3333333333333333
  },
  "decision_rule": "FAKE if at least 2 of 3 regional models predict FAKE",
  "accuracy": 0.5969387755102041,
  "balanced_accuracy": 0.5995310057321521,
  "precision_fake": 0.5701754385964912,
  "recall_fake": 0.6842105263157895,
  "specificity_real": 0.5148514851485149,
  "f1_fake": 0.6220095693779905,
  "roc_auc_vote_score": 0.6245961438249088,
  "average_precision_vote_score": 0.5702613259545695,
  "mcc": 0.20167474934828117,
  "confusion_matrix": {
    "tn": 52,
    "fp": 49,
    "fn": 30,
    "tp": 65
  }
}


In [8]:
# 8) Ortak test kohortu üzerinde bölgesel modelleri ve füzyonu karsilastir

def binary_metrics(y, pred, score=None):
    cm_local = confusion_matrix(y, pred, labels=[0, 1])
    tn_, fp_, fn_, tp_ = cm_local.ravel()
    out = {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision_fake": precision_score(y, pred, zero_division=0),
        "recall_fake": recall_score(y, pred, zero_division=0),
        "specificity_real": tn_ / (tn_ + fp_) if (tn_ + fp_) else np.nan,
        "f1_fake": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
        "roc_auc_score": np.nan,
        "average_precision_score": np.nan,
    }
    if score is not None:
        out["roc_auc_score"] = roc_auc_score(y, score)
        out["average_precision_score"] = average_precision_score(y, score)
    return out

comparison_rows = [
    {
        "system": "Eyebrow VGG16+SVM",
        "score_type": "saved fake_probability",
        **binary_metrics(
            y_true,
            fused["pred_eyebrow"].to_numpy(int),
            fused["eyebrow_fake_probability"].to_numpy(float),
        ),
    },
    {
        "system": "Mouth VGG16+SVM",
        "score_type": "saved decision_score_fake",
        **binary_metrics(
            y_true,
            fused["pred_mouth"].to_numpy(int),
            fused["mouth_decision_score_fake"].to_numpy(float),
        ),
    },
    {
        "system": "Eye VGG16+SVM",
        "score_type": "saved fake_probability",
        **binary_metrics(
            y_true,
            fused["pred_eye"].to_numpy(int),
            fused["eye_fake_probability"].to_numpy(float),
        ),
    },
    {
        "system": "3-Region Majority-Vote Fusion",
        "score_type": "vote_fraction_0_to_1",
        **binary_metrics(y_true, y_pred, vote_score),
    },
]

region_comparison = pd.DataFrame(comparison_rows)
display(region_comparison.round(6))

,system,score_type,accuracy,balanced_accuracy,precision_fake,recall_fake,specificity_real,f1_fake,mcc,roc_auc_score,average_precision_score
0,Eyebrow VGG16+SVM,saved fake_probability,0.576531,0.577228,0.558824,0.600000,0.554455,0.578680,0.154512,0.600417,0.597534
1,Mouth VGG16+SVM,saved decision_score_fake,0.566327,0.567952,0.546296,0.621053,0.514851,0.581281,0.136553,0.626889,0.565702
2,Eye VGG16+SVM,saved fake_probability,0.576531,0.580042,0.550000,0.694737,0.465347,0.613953,0.164199,0.585253,0.578336
3,3-Region Majority-Vote Fusion,vote_fraction_0_to_1,0.596939,0.599531,0.570175,0.684211,0.514851,0.622010,0.201675,0.624596,0.570261


In [9]:
# 9) SSOT uyumlu çıktı klasörünü olustur ve atomik yazma yardimcilari

now_tr = datetime.now(ZoneInfo("Europe/Istanbul"))
RUN_ID = (
    now_tr.strftime("%Y%m%d_%H%M")
    + "_multi_region_vgg16_svm_majority_vote_seed42"
)
RUN_DIR = FUSION_ROOT / RUN_ID

if RUN_DIR.exists():
    raise FileExistsError(
        f"Run klasörü zaten var ve üzerine yazilmayacak: {RUN_DIR}\n"
        "Yeni bir dakikada tekrar çalistirin."
    )

subdirs = {
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "logs": RUN_DIR / "logs",
    "artifacts": RUN_DIR / "artifacts",
    "checkpoints": RUN_DIR / "checkpoints",
}

RUN_DIR.mkdir(parents=True, exist_ok=False)
for p in subdirs.values():
    p.mkdir(parents=False, exist_ok=False)

def atomic_write_text(path: Path, text: str):
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def atomic_write_json(path: Path, obj):
    atomic_write_text(
        path,
        json.dumps(obj, indent=2, ensure_ascii=False, default=str) + "\n"
    )

def atomic_write_csv(path: Path, df: pd.DataFrame):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    check = pd.read_csv(tmp, encoding="utf-8-sig")
    assert len(check) == len(df), f"CSV atomik dogrulama basarisiz: {path}"
    os.replace(tmp, path)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)

RUN_ID : 20260810_0019_multi_region_vgg16_svm_majority_vote_seed42
RUN_DIR: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/füzyon sonuçları/20260810_0019_multi_region_vgg16_svm_majority_vote_seed42


In [10]:
# 10) Predictions, metrics, config, provenance ve audit dosyalarini kaydet

fusion_output_cols = [
    "frame_key",
    "true_label", "true_class",
    "pred_eyebrow", "pred_mouth", "pred_eye",
    "eyebrow_fake_probability", "eyebrow_decision_score",
    "mouth_decision_score_fake",
    "eye_fake_probability", "eye_decision_score",
    "fake_vote_count", "fusion_vote_score",
    "fusion_predicted_label", "fusion_predicted_class",
    "correct",
    "eyebrow_image_path", "mouth_image_path", "eye_image_path",
    "mouth_source_sample_id", "eye_source_sample_id",
]

atomic_write_csv(
    subdirs["predictions"] / "fused_test_predictions.csv",
    fused[fusion_output_cols].copy(),
)

atomic_write_json(
    subdirs["metrics"] / "fusion_metrics.json",
    fusion_metrics,
)

atomic_write_csv(
    subdirs["metrics"] / "region_comparison_common_cohort.csv",
    region_comparison,
)

report_dict = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["REAL", "FAKE"],
    output_dict=True,
    zero_division=0,
)
atomic_write_json(
    subdirs["metrics"] / "classification_report.json",
    report_dict,
)

cm_df = pd.DataFrame(
    cm,
    index=["true_REAL", "true_FAKE"],
    columns=["pred_REAL", "pred_FAKE"],
)
atomic_write_csv(
    subdirs["metrics"] / "confusion_matrix.csv",
    cm_df.reset_index(names="true_class"),
)

matching_audit = {
    "source_counts": {
        "eyebrow_raw": int(len(eyebrow)),
        "mouth_raw": int(len(mouth)),
        "eye_raw": int(len(eye)),
        "mouth_face0": int(len(mouth0)),
        "eye_face0": int(len(eye0)),
    },
    "excluded_nonzero_faces": {
        "mouth": int(len(mouth_extra_faces)),
        "eye": int(len(eye_extra_faces)),
    },
    "eyebrow_missing_in_mouth_face0": int(len(missing_in_mouth)),
    "eyebrow_missing_in_eye_face0": int(len(missing_in_eye)),
    "matched_common_test_samples": int(len(fused)),
    "all_eyebrow_samples_matched": bool(len(fused) == len(eyebrow)),
    "ground_truth_agreement_all_regions": True,
    "matching_key": "frame_key derived from real_test_XXXXX / fake_test_XXXXX",
    "face_policy": "face_index=0 for Mouth and Eye",
}
atomic_write_json(
    subdirs["artifacts"] / "matching_audit.json",
    matching_audit,
)

if len(mouth_extra_faces):
    atomic_write_csv(
        subdirs["artifacts"] / "excluded_mouth_nonzero_faces.csv",
        mouth_extra_faces,
    )
if len(eye_extra_faces):
    atomic_write_csv(
        subdirs["artifacts"] / "excluded_eye_nonzero_faces.csv",
        eye_extra_faces,
    )

source_provenance = {
    "no_drive_ids_hardcoded": True,
    "path_resolution": "name-based, Unicode-normalized",
    "source_runs": {
        "eyebrow": {
            "run_name": EYEBROW_RUN_NAME,
            "run_path": str(EYEBROW_RUN),
            "prediction_file": str(eyebrow_pred_path),
            "prediction_sha256": sha256_file(eyebrow_pred_path),
        },
        "mouth": {
            "run_name": MOUTH_RUN_NAME,
            "run_path": str(MOUTH_RUN),
            "prediction_file": str(mouth_pred_path),
            "prediction_sha256": sha256_file(mouth_pred_path),
            "test_features_file": str(mouth_feature_path),
            "test_features_sha256": sha256_file(mouth_feature_path),
        },
        "eye": {
            "run_name": EYE_RUN_NAME,
            "run_path": str(EYE_RUN),
            "prediction_file": str(eye_pred_path),
            "prediction_sha256": sha256_file(eye_pred_path),
        },
    },
    "target_root": str(FUSION_ROOT),
    "created_run_dir": str(RUN_DIR),
}
atomic_write_json(RUN_DIR / "source_provenance.json", source_provenance)

fusion_config = {
    "schema_version": "ssot-v2.0-fusion",
    "run_id": RUN_ID,
    "seed": SEED,
    "region": "multi_region",
    "source_runs": {
        "eyebrow": EYEBROW_RUN_NAME,
        "mouth": MOUTH_RUN_NAME,
        "eye": EYE_RUN_NAME,
    },
    "fusion": {
        "method": "equal_weight_3way_majority_vote_late_fusion",
        "retraining": False,
        "meta_model": None,
        "calibration": None,
        "weights": {"eyebrow": 1/3, "mouth": 1/3, "eye": 1/3},
        "positive_class": "FAKE",
        "final_rule": "fake_vote_count >= 2",
        "vote_score": "(pred_eyebrow + pred_mouth + pred_eye) / 3",
    },
    "matching": {
        "key": "frame_key",
        "face_policy": "Mouth/Eye face_index=0",
        "require_complete_eyebrow_coverage": True,
        "require_ground_truth_agreement": True,
    },
    "output_dir": str(RUN_DIR),
}
atomic_write_text(
    RUN_DIR / "config_resolved.yaml",
    json.dumps(fusion_config, indent=2, ensure_ascii=False) + "\n",
)

atomic_write_text(
    subdirs["checkpoints"] / "README.txt",
    "No checkpoint was created because this run performs no model training, "
    "fine-tuning, calibration fitting, or meta-model fitting.\n"
)

In [11]:
# 11) Grafikler - %100 English, min short side >= 600 px, PNG + PDF

def save_figure_verified(fig, stem: str):
    png_target = subdirs["figures"] / f"{stem}.png"
    pdf_target = subdirs["figures"] / f"{stem}.pdf"

    png_tmp = subdirs["figures"] / f".{stem}.tmp.png"
    pdf_tmp = subdirs["figures"] / f".{stem}.tmp.pdf"

    fig.savefig(png_tmp, dpi=150, bbox_inches="tight")
    fig.savefig(pdf_tmp, bbox_inches="tight")

    with Image.open(png_tmp) as im:
        assert min(im.size) >= 600, (
            f"Figure resolution too low: {stem} -> {im.size}"
        )

    os.replace(png_tmp, png_target)
    os.replace(pdf_tmp, pdf_target)
    plt.close(fig)

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
im = ax.imshow(cm)
ax.set_title(
    "3-Region Late Fusion - Confusion Matrix",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
ax.set_xticks([0, 1], labels=["REAL", "FAKE"])
ax.set_yticks([0, 1], labels=["REAL", "FAKE"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
save_figure_verified(fig, "confusion_matrix")

# ROC
fpr, tpr, _ = roc_curve(y_true, vote_score)
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    fpr, tpr, linewidth=2,
    label=f"Fusion ROC-AUC = {fusion_metrics['roc_auc_vote_score']:.3f}"
)
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Chance")
ax.set_title(
    "3-Region Late Fusion - ROC Curve",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(frameon=True, loc="lower right")
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure_verified(fig, "roc_curve")

# Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(y_true, vote_score)
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    recall_curve,
    precision_curve,
    linewidth=2,
    label=(
        "Average Precision = "
        f"{fusion_metrics['average_precision_vote_score']:.3f}"
    ),
)
ax.set_title(
    "3-Region Late Fusion - Precision-Recall Curve",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.legend(frameon=True, loc="lower left")
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure_verified(fig, "precision_recall_curve")

# Comparison
plot_df = region_comparison.set_index("system")[["accuracy", "f1_fake"]]
fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
plot_df.plot(kind="bar", ax=ax)
ax.set_title(
    "Regional Models vs. 3-Region Late Fusion",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_xlabel("System", fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1)
ax.legend(["Accuracy", "F1 (FAKE)"], frameon=True)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
save_figure_verified(fig, "regional_vs_fusion_comparison")

print("All figures saved and verified (short side >= 600 px).")

All figures saved and verified (short side >= 600 px).


In [12]:
# 12) Environment, quality gates, README, log ve output manifest

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn_version,
    "matplotlib": plt.matplotlib.__version__,
    "run_timezone": "Europe/Istanbul",
    "seed": SEED,
}
atomic_write_json(RUN_DIR / "environment.json", environment)

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    capture_output=True,
    text=True,
).stdout
atomic_write_text(RUN_DIR / "requirements_lock.txt", freeze)

numeric_cols = [
    "eyebrow_fake_probability",
    "eyebrow_decision_score",
    "mouth_decision_score_fake",
    "eye_fake_probability",
    "eye_decision_score",
    "fusion_vote_score",
]
numeric_ok = np.isfinite(
    fused[numeric_cols].to_numpy(dtype=float)
).all()

quality_gates = {
    "source_path_resolution_by_name": "PASSED",
    "no_hardcoded_drive_ids": "PASSED",
    "source_files_read_only": "PASSED",
    "source_prediction_schema": "PASSED",
    "mouth_csv_npz_row_alignment": "PASSED",
    "canonical_frame_key_parse": "PASSED",
    "face0_uniqueness": "PASSED",
    "complete_eyebrow_cross_region_coverage": "PASSED",
    "ground_truth_label_agreement": "PASSED",
    "no_retraining_or_meta_model": "PASSED",
    "test_rule_fixed_before_evaluation": "PASSED",
    "nan_inf_numeric_check": "PASSED" if numeric_ok else "FAILED",
    "figure_language": "ENGLISH",
    "figure_min_short_side_600px": "PASSED",
    "output_atomic_writes": "PASSED",
}
if "FAILED" in quality_gates.values():
    raise RuntimeError(f"Quality gate failed: {quality_gates}")

atomic_write_json(RUN_DIR / "quality_gates.json", quality_gates)

readme_lines = [
    "3-REGION VGG16 + SVM LATE FUSION",
    "================================",
    "",
    f"Run ID: {RUN_ID}",
    "Status: COMPLETED AND VALIDATED",
    "",
    "SOURCE RUNS",
    "-----------",
    f"Eyebrow : {EYEBROW_RUN_NAME}",
    f"Mouth   : {MOUTH_RUN_NAME}",
    f"Eye     : {EYE_RUN_NAME}",
    "",
    "FUSION METHOD",
    "-------------",
    "Method         : Equal-weight 3-way Majority-Vote Late Fusion",
    "New training   : NO",
    "Meta-model     : NO",
    "Calibration fit: NO",
    "Weights        : Eyebrow=1/3, Mouth=1/3, Eye=1/3",
    "Final rule     : FAKE if at least 2 of 3 regional models predict FAKE",
    "",
    "MATCHING",
    "--------",
    "Canonical key        : real_test_XXXXX / fake_test_XXXXX",
    "Mouth/Eye face policy: face_index=0",
    f"Matched test samples : {len(fused)}",
    f"Mouth extra faces excluded from fusion: {len(mouth_extra_faces)}",
    f"Eye extra faces excluded from fusion  : {len(eye_extra_faces)}",
    "Ground-truth agreement: PASS",
    "",
    "FINAL TEST RESULTS",
    "------------------",
    f"Accuracy          : {fusion_metrics['accuracy']:.6f}",
    f"Balanced Accuracy : {fusion_metrics['balanced_accuracy']:.6f}",
    f"Precision (FAKE)  : {fusion_metrics['precision_fake']:.6f}",
    f"Recall (FAKE)     : {fusion_metrics['recall_fake']:.6f}",
    f"Specificity (REAL): {fusion_metrics['specificity_real']:.6f}",
    f"F1-score (FAKE)   : {fusion_metrics['f1_fake']:.6f}",
    f"ROC-AUC (vote score): {fusion_metrics['roc_auc_vote_score']:.6f}",
    (
        "Average Precision (vote score): "
        f"{fusion_metrics['average_precision_vote_score']:.6f}"
    ),
    f"MCC               : {fusion_metrics['mcc']:.6f}",
    "",
    "CONFUSION MATRIX",
    "----------------",
    f"TN={tn}, FP={fp}, FN={fn}, TP={tp}",
    "",
    "IMPORTANT",
    "---------",
    (
        "The fusion uses only saved regional model decisions. "
        "The Mouth source run does not provide a saved fake_probability "
        "comparable to the Eyebrow/Eye probabilities."
    ),
    (
        "Therefore raw SVM decision scores are NOT averaged with probabilities "
        "and no new calibration/model is fitted."
    ),
    "",
    "OUTPUT CONTENTS",
    "---------------",
    "predictions/ : Fused sample-level test predictions",
    "metrics/     : Final metrics, classification report, region comparison",
    "figures/     : English PNG/PDF figures, verified >=600 px short side",
    "artifacts/   : Matching audit and excluded non-zero-face rows",
    "logs/        : Run log",
    "checkpoints/ : README only; no checkpoint because no training",
]
readme = "\n".join(readme_lines) + "\n"
atomic_write_text(RUN_DIR / "README_FINAL.txt", readme)

log_text = (
    f"[{datetime.now(ZoneInfo('Europe/Istanbul')).isoformat()}] "
    f"COMPLETED {RUN_ID}; matched={len(fused)}; "
    f"accuracy={fusion_metrics['accuracy']:.6f}; "
    f"f1={fusion_metrics['f1_fake']:.6f}\n"
)
atomic_write_text(subdirs["logs"] / "fusion.log", log_text)

manifest_rows = []
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file() and p.name not in {
        "output_manifest.csv", "output_manifest.json"
    }:
        manifest_rows.append({
            "relative_path": str(p.relative_to(RUN_DIR)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(manifest_rows)
atomic_write_csv(RUN_DIR / "output_manifest.csv", manifest_df)
atomic_write_json(RUN_DIR / "output_manifest.json", manifest_rows)

print(readme)
print("Saved to:", RUN_DIR)

3-REGION VGG16 + SVM LATE FUSION

Run ID: 20260810_0019_multi_region_vgg16_svm_majority_vote_seed42
Status: COMPLETED AND VALIDATED

SOURCE RUNS
-----------
Eyebrow : VGG16_FeatureExtractor_SVM_Kas
Mouth   : 20260808_1240_mouth_vgg16_svm_seed42
Eye     : 20260808_1124_eye_vgg16_svm_seed42

FUSION METHOD
-------------
Method         : Equal-weight 3-way Majority-Vote Late Fusion
New training   : NO
Meta-model     : NO
Calibration fit: NO
Weights        : Eyebrow=1/3, Mouth=1/3, Eye=1/3
Final rule     : FAKE if at least 2 of 3 regional models predict FAKE

MATCHING
--------
Canonical key        : real_test_XXXXX / fake_test_XXXXX
Mouth/Eye face policy: face_index=0
Matched test samples : 196
Mouth extra faces excluded from fusion: 10
Eye extra faces excluded from fusion  : 10
Ground-truth agreement: PASS

FINAL TEST RESULTS
------------------
Accuracy          : 0.596939
Balanced Accuracy : 0.599531
Precision (FAKE)  : 0.570175
Recall (FAKE)     : 0.684211
Specificity (REAL): 0.514851
F1

In [13]:
# 13) Son kontrol - run klasöründeki tüm çıktıları listele

for p in sorted(RUN_DIR.rglob("*")):
    print(p.relative_to(RUN_DIR))

README_FINAL.txt
artifacts
artifacts/excluded_eye_nonzero_faces.csv
artifacts/excluded_mouth_nonzero_faces.csv
artifacts/matching_audit.json
checkpoints
checkpoints/README.txt
config_resolved.yaml
environment.json
figures
figures/confusion_matrix.pdf
figures/confusion_matrix.png
figures/precision_recall_curve.pdf
figures/precision_recall_curve.png
figures/regional_vs_fusion_comparison.pdf
figures/regional_vs_fusion_comparison.png
figures/roc_curve.pdf
figures/roc_curve.png
logs
logs/fusion.log
metrics
metrics/classification_report.json
metrics/confusion_matrix.csv
metrics/fusion_metrics.json
metrics/region_comparison_common_cohort.csv
output_manifest.csv
output_manifest.json
predictions
predictions/fused_test_predictions.csv
quality_gates.json
requirements_lock.txt
source_provenance.json
